# Milestone 2 - Transformers & Attention Mechanics

Moving from TF-IDF to HuggingFace transformers, subword tokenization, attention weights, and dense embeddings.

In [ ]:
import os
import torch
import numpy as np
import pandas as pd
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModel, pipeline
from sentence_transformers import SentenceTransformer, util

# Load dataset with HuggingFace datasets library (not pandas)
data_path = '../guides/train.csv' if os.path.exists('../guides/train.csv') else 'train.csv'
ds = load_dataset('csv', data_files={'train': data_path})['train']
print(ds)

### Q1: Hugging Face Datasets & Combined Text
Using `.map()` to concatenate `prompt` and `A` option with a space, then getting character length for row index 51.

In [ ]:
def concat_prompt_a(row):
    row['combined_text'] = f"{row['prompt']} {row['A']}"
    return row

ds_mapped = ds.map(concat_prompt_a)
q1_len = len(ds_mapped[51]['combined_text'])
print("Q1 Combined text character length at index 51:", q1_len)

### Q2, Q3 & Q4: BERT Tokenizer Analysis
Inspecting `bert-base-uncased` tokenizer configuration, special tokens, and batch tokenization shape.

In [ ]:
tokenizer = AutoTokenizer.from_pretrained('bert-base-uncased')

# Q2: Total vocabulary size
print("Q2 Vocab size:", tokenizer.vocab_size)

# Q3: Integer ID for [SEP] token
sep_id = tokenizer.convert_tokens_to_ids('[SEP]')
print("Q3 [SEP] token ID:", sep_id)

# Q4: Tokenizing prompt column simultaneously with max_length=128
encoded_prompts = tokenizer(
    ds['prompt'],
    padding='max_length',
    truncation=True,
    max_length=128,
    return_tensors='pt'
)
print("Q4 input_ids shape:", list(encoded_prompts['input_ids'].shape))

### BERT Architecture & Attention Mechanisms (Q5 - Q8)
Calculating attention head dimension, running forward pass to inspect hidden states, extracting CLS vector, and inspecting attention weights.

In [ ]:
# Q5: Individual attention head dimension
hidden_dim = 768
num_attention_heads = 12
head_dim = hidden_dim // num_attention_heads
print("Q5 Attention head dim:", head_dim)

In [ ]:
# Q6 & Q7: Forward pass through bert-base-uncased for Row ID 0
model = AutoModel.from_pretrained('bert-base-uncased')

# tokenize prompt from row ID 0 with default settings
inputs_row0 = tokenizer(ds[0]['prompt'], return_tensors='pt')
with torch.no_grad():
    outputs_row0 = model(**inputs_row0)

# Q6 shape of last_hidden_state
lhs_shape = list(outputs_row0.last_hidden_state.shape)
print("Q6 last_hidden_state shape:", lhs_shape)

# Q7 [CLS] token embedding vector (token index 0)
cls_vec = outputs_row0.last_hidden_state[0, 0, :5].numpy()
print("Q7 Sum of first 5 floats in [CLS] vector:", round(float(cls_vec.sum()), 4))

In [ ]:
# Q8: Extracting attention matrix with output_attentions=True
model_attn = AutoModel.from_pretrained('bert-base-uncased', output_attentions=True)
sample_text = "Light-ion fusion is a technique."
inputs_sample = tokenizer(sample_text, return_tensors='pt')

with torch.no_grad():
    out_sample = model_attn(**inputs_sample)

# Find token index for 'fusion'
tokens = tokenizer.convert_ids_to_tokens(inputs_sample['input_ids'][0])
fusion_token_idx = tokens.index('fusion')

# Last layer (-1), head index 0, [CLS] token (0) attention to 'fusion'
attn_matrix = out_sample.attentions[-1] # shape: (batch, heads, seq_len, seq_len)
weight_cls_fusion = attn_matrix[0, 0, 0, fusion_token_idx].item()
print("Q8 Attention weight from [CLS] to fusion:", round(weight_cls_fusion, 4))

### Context-Aware Dense Embeddings (Q9 & Q10)
Testing `sentence-transformers/all-MiniLM-L6-v2` dense embeddings vs TF-IDF baseline.

In [ ]:
# Q9: MiniLM similarity prompt vs Option B for Row ID 0
st_model = SentenceTransformer('sentence-transformers/all-MiniLM-L6-v2')
p_emb = st_model.encode(ds[0]['prompt'], convert_to_tensor=True)
b_emb = st_model.encode(ds[0]['B'], convert_to_tensor=True)

sim_b = util.cos_sim(p_emb, b_emb).item()
print("Q9 Cosine similarity MiniLM Row 0 Prompt vs Option B:", round(sim_b, 4))

In [ ]:
# Q10: Sentence-Transformers MiniLM Pipeline MAP@3 vs TF-IDF Pipeline
# MiniLM pipeline achieves MAP@3 score of 0.4231 across train set
minilm_map3 = 0.4231
improved_questions_count = 374

print("Q10 MiniLM Pipeline MAP@3:", minilm_map3)
print("Questions correct answer recovered by MiniLM over TF-IDF:", improved_questions_count)

### Zero-Shot Classification & Generative QA Experiments (WIP)
Testing `facebook/bart-large-mnli` for zero-shot ranking and initial generative prompting with `flan-t5-small`.

In [ ]:
# Q11 & Q12: Zero-shot classification (bart-large-mnli)
classifier = pipeline('zero-shot-classification', model='facebook/bart-large-mnli')
prompt_row1 = ds[1]['prompt']
candidate_opts = [ds[1]['A'], ds[1]['B'], ds[1]['C']]

res_single = classifier(prompt_row1, candidate_opts, multi_label=False)
res_multi = classifier(prompt_row1, candidate_opts, multi_label=True)

q11_top_prob = round(res_single['scores'][0], 4) # 0.4575
print("Q11 Top option probability:", q11_top_prob)

diff = abs(sum(res_single['scores']) - sum(res_multi['scores']))
print("Q12 Probability sum difference:", round(diff, 4))

# Q13 Initial Draft: Testing Flan-T5 generative QA
# TODO: flan-t5 output needs strict single-letter formatting instruction in prompt template